# Module 7 Homework: API Chain Project
# E-Commerce Keyword Trend Analyzer

This project uses an API chain to connect keyword trend research with Amazon marketplace data and product reviews.

# Google Keyword API

This section uses a RapidAPI API to pull the Google top 3 keywords based on your search criteria.

In [15]:
import requests
import pandas as pd
from google.colab import userdata

RAPID_API_KEY = userdata.get("RAPID_API_KEY")

In [16]:
url = "https://google-keyword-insight1.p.rapidapi.com/topkeys/"

querystring = {
    "keyword": "Seasonal Allergy",
    "location": "US",
    "lang": "en"
}

headers = {
    "x-rapidapi-key": RAPID_API_KEY,
    "x-rapidapi-host": "google-keyword-insight1.p.rapidapi.com",
    "Content-Type": "application/json"
}

response = requests.get(url, headers=headers, params=querystring)

keyword_data = response.json()

print("Google keyword search completed successfully.")
print("Number of keywords returned:", len(keyword_data))

Google keyword search completed successfully.
Number of keywords returned: 10


In [17]:
top_keywords = [k["text"] for k in keyword_data[:3]]
print("Top keywords:", top_keywords)

Top keywords: ['hay fever seasonal allergies', 'seasonal allergy', 'symptoms of ragweed allergy']


# Amazon Product Search

This section uses a RapidAPI API to perform an Amazon Product Search on the top 3 keywords from the previous section.

In [18]:
all_rows = []

for keyword in top_keywords:
    url = "https://real-time-amazon-data.p.rapidapi.com/search"

    querystring = {
        "query": keyword,
        "page": "1",
        "country": "US",
        "sort_by": "RELEVANCE",
        "product_condition": "ALL",
        "is_prime": "false",
        "deals_and_discounts": "NONE"
    }

    headers = {
        "x-rapidapi-key": RAPID_API_KEY,
        "x-rapidapi-host": "real-time-amazon-data.p.rapidapi.com",
        "Content-Type": "application/json"
    }

    response = requests.get(url, headers=headers, params=querystring)
    amazon_products = response.json()

    if "data" not in amazon_products or "products" not in amazon_products["data"]:
        print(f"No products returned for keyword: {keyword}")
        continue

    products = amazon_products["data"]["products"]

    if not products:
        print(f"No products found for keyword: {keyword}")
        continue

    ranked_products = sorted(
        products,
        key=lambda p: (
            float(p.get("product_star_rating", 0) or 0),
            int(p.get("product_num_ratings", 0) or 0)
        ),
        reverse=True
    )

    top_3_products = ranked_products[:3]

    print(f"Processed keyword: {keyword}")

    for p in top_3_products:
        all_rows.append({
            "google_keyword": keyword,
            "asin": p.get("asin"),
            "product_title": p.get("product_title"),
            "product_price": p.get("product_price"),
            "unit_price": p.get("unit_price"),
            "unit_count": p.get("unit_count"),
            "currency": p.get("currency"),
            "product_star_rating": p.get("product_star_rating"),
            "product_num_ratings": p.get("product_num_ratings"),
            "product_url": p.get("product_url"),
            "product_photo": p.get("product_photo"),
            "product_num_offers": p.get("product_num_offers"),
            "product_minimum_offer_price": p.get("product_minimum_offer_price"),
            "is_best_seller": p.get("is_best_seller"),
            "is_amazon_choice": p.get("is_amazon_choice"),
            "is_prime": p.get("is_prime"),
            "sales_volume": p.get("sales_volume"),
            "delivery": p.get("delivery"),
            "product_badge": p.get("product_badge"),
            "product_byline": p.get("product_byline")
        })

Processed keyword: hay fever seasonal allergies
Processed keyword: seasonal allergy
Processed keyword: symptoms of ragweed allergy


In [19]:
final_df = pd.DataFrame(all_rows)
final_df

,google_keyword,asin,product_title,product_price,unit_price,unit_count,currency,product_star_rating,product_num_ratings,product_url,product_photo,product_num_offers,product_minimum_offer_price,is_best_seller,is_amazon_choice,is_prime,sales_volume,delivery,product_badge,product_byline
0,hay fever seasonal allergies,B005TM2606,MAJOR Banophen Diphenhydramine HCI 25 mg Minit...,$6.50,$0.27,24,USD,5,5,https://www.amazon.com/dp/B005TM2606,https://m.media-amazon.com/images/I/610pY+sEgB...,9,$6.28,False,False,False,None,"FREE delivery Thu, Mar 19 on $35 of items ship...",None,None
1,hay fever seasonal allergies,B07HD8SFNT,Amazon Basic Care Allergy Relief Diphenhydrami...,$4.47,$0.04,112,USD,4.8,52702,https://www.amazon.com/dp/B07HD8SFNT,https://m.media-amazon.com/images/I/61BBvtpWAK...,1,$4.47,False,False,False,None,"FREE delivery Thu, Mar 19 on $35 of items ship...",None,100 Count (Pack of 1)
2,hay fever seasonal allergies,B00GA9AVH2,"Benadryl Ultratabs Allergy Medicine, 25 mg Dip...",$14.88,$0.15,99,USD,4.8,31252,https://www.amazon.com/dp/B00GA9AVH2,https://m.media-amazon.com/images/I/71pQCG+ytt...,1,$14.88,False,False,False,None,"FREE delivery Thu, Mar 19 on $35 of items ship...",None,100 Count (Pack of 1)
3,seasonal allergy,B07HD8SFNT,Amazon Basic Care Allergy Relief Diphenhydrami...,$4.47,$0.04,112,USD,4.8,52702,https://www.amazon.com/dp/B07HD8SFNT,https://m.media-amazon.com/images/I/61BBvtpWAK...,1,$4.47,False,False,False,None,"FREE delivery Thu, Mar 19 on $35 of items ship...",None,100 Count (Pack of 1)
4,seasonal allergy,B0014CX4YW,"Benadryl Liqui-Gels Allergy Medicine, Dye-Free...",$6.99,$0.29,24,USD,4.8,13482,https://www.amazon.com/dp/B0014CX4YW,https://m.media-amazon.com/images/I/814uFgWtLd...,1,$6.99,False,False,False,None,"FREE delivery Thu, Mar 19 on $35 of items ship...",None,24 Count
5,seasonal allergy,B078BBYNLC,"Claritin Allergy Medicine for Adults, 24-Hour ...",$44.97,$0.45,100,USD,4.8,10548,https://www.amazon.com/dp/B078BBYNLC,https://m.media-amazon.com/images/I/81Lyn9xJdW...,1,$44.97,False,False,False,None,"FREE delivery Thu, Mar 19Or fastest delivery T...",None,100 Count (Pack of 1)
6,symptoms of ragweed allergy,B0GH8Y6DYD,CalmCo Adult Liquid Allergy Relief – Pre-Measu...,$24.95,$1.39,18,USD,5,6,https://www.amazon.com/dp/B0GH8Y6DYD,https://m.media-amazon.com/images/I/81BRaXoByg...,1,$24.95,False,False,False,None,"FREE delivery Thu, Mar 19 on $35 of items ship...",None,18 Count
7,symptoms of ragweed allergy,B09XRT9ZNQ,Boiron Ambrosia Artemisiaefolia 30C Homeopathi...,$18.30,$0.08,229,USD,5,5,https://www.amazon.com/dp/B09XRT9ZNQ,https://m.media-amazon.com/images/I/81C239DLnl...,1,$18.30,False,False,False,None,"FREE delivery Thu, Mar 19 on $35 of items ship...",None,80 Count (Pack of 3)
8,symptoms of ragweed allergy,B00H4HGQEG,WishGarden Herbs Kick-Ass Allergy - Plant-Base...,$13.99,$21.20,1,USD,5,4,https://www.amazon.com/dp/B00H4HGQEG,https://m.media-amazon.com/images/I/61NmFVjqRE...,1,$13.99,False,False,False,None,"$5.59 delivery Mon, Mar 23Only 8 left in stock...",None,Liquid0.66 Fl Oz (Pack of 1)


# Google Sheets Export

This section uses exports the Amazon product search results to a Google Sheets document.

In [20]:
!pip install gspread gspread-dataframe

In [23]:
from google.colab import auth
auth.authenticate_user()

import gspread
from google.auth import default
from gspread_dataframe import set_with_dataframe

creds, _ = default()
gc = gspread.authorize(creds)

In [24]:
spreadsheet_name = "BAN3110_Ecommerce_Keyword_Analyzer"

try:
    sh = gc.open(spreadsheet_name)
except:
    sh = gc.create(spreadsheet_name)

worksheet = sh.sheet1
worksheet.clear()
set_with_dataframe(worksheet, final_df)

print("Data exported to Google Sheets successfully.")
print("Spreadsheet URL:", sh.url)

Data exported to Google Sheets successfully.
Spreadsheet URL: https://docs.google.com/spreadsheets/d/16DCLebwUlnH8zE-yBIGiiWwo6ojS0xJ41UvOYRIb1tM
